> **SOLUTIONS notebook.** Answers are filled into the `#answer` cells below; Try the exercises yourself first.

This file is part of the CRISPRsummerschool 2026 exercises

Copyright (c) 2023-26 Christian Anthon & 2026 Gül Sude Demircan

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, version 3.
# Extracting features from the deep learning results
Below you will find the second exercise.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RTH-tools/CRISPRsummerschool/blob/main/2026/CRISPR/exercise/crispr_2026_crispr_exercise2.ipynb)

Deep learning does not easily lend itself to extraction of feature importance, like in the example of CRISPR where one could wish to know the importance of *e.g.* the first nucleotide of the NGG pam for the efficiency of the guide. In this exercise we will look at a way around this problem by masking out parts of the input sequence or of the energy parameter from the model input.


## basic code definitions
Enter the cell below and press play or Ctrl+Enter in the block below to execute. You should see the message "Data loaded" printed after execution.

In [1]:
#!/usr/bin/env python3
# CRISPRsummerschool 2026 -- PyTorch version
import os
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


eLENGTH30 = 30
eDEPTH = 4

# Function to onehot encode the data
def onehot(x):
    z = list()
    for y in list(x):
        if y in "Aa":
            z.append(0)
        elif y in "Cc":
            z.append(1)
        elif y in "Gg":
            z.append(2)
        elif y in "TtUu":
            z.append(3)
        else:
            print("Non-ATGCU character in", x)
            raise Exception
    return z

# Function to set the data into the appropriate format
def set_data(DX, s, mask=None):
    # mask should be a list of length len(s) of 1s and 0s: positions where mask
    # is 0 are onehot encoded, positions with 1 are masked out (left as zeros).
    if s is None:
        return
    assert(mask == None or (type(mask) is list and len(mask) == len(s)))
    if type(mask) is list:
        for j, x in enumerate(onehot(s)):
            if mask[j] == 0:
                DX[j][x] = 1
    else:
        for j, x in enumerate(onehot(s)):
            DX[j][x] = 1

# Preprocessing function for the sequence data
def preprocess_seq(data, mask=None, use_dgb=True):
    DATA_X30 = np.zeros((len(data), eLENGTH30, eDEPTH), dtype=np.float32)  # onehot
    DATA_G = np.zeros((len(data), 1), dtype=np.float32)  # deltaGb
    DATA_Y = np.zeros((len(data)), dtype=np.float32)  # efficiency

    for l, d in enumerate(data):
        set_data(DATA_X30[l], d[1], mask)
        if use_dgb:
            DATA_G[l] = -d[2]
        DATA_Y[l] = d[3]
    return (DATA_X30, DATA_G, DATA_Y)


# Convert numpy arrays to torch tensors on `device`.
# NOTE: the one-hot stays (N, 30, 4) here; the model permutes it to
#       (N, 4, 30) internally, because PyTorch Conv1d expects the layout
#       (batch, channels, length).
def to_tensors(x30, g, y):
    return (torch.from_numpy(x30).to(device),
            torch.from_numpy(g).to(device),
            torch.from_numpy(y).to(device))

def evaluate(model, data):
    """Return (mse, mae) on a (Xc, Xg, y) split. Runs in eval mode (dropout OFF)."""
    Xc, Xg, y = data
    model.eval()
    with torch.no_grad():
        pred = model(Xc, Xg).squeeze(-1)          # (N, 1) -> (N,)
        mse = torch.mean((pred - y) ** 2).item()
        mae = torch.mean(torch.abs(pred - y)).item()
    return mse, mae

def train(model, train_data, val_data, epochs=200, batch_size=64, lr=1e-3,
          patience=25, min_delta=0.1, verbose=True):
    """Mini-batch training with early stopping and restore-best-weights."""
    Xc, Xg, y = train_data
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait, history = float("inf"), None, 0, []
    n = Xc.shape[0]
    for epoch in range(epochs):
        model.train()                             # dropout ON
        perm = torch.randperm(n, device=Xc.device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            pred = model(Xc[idx], Xg[idx]).squeeze(-1)   # (B,1) -> (B,): MUST squeeze
            loss = loss_fn(pred, y[idx])
            loss.backward()
            optimizer.step()
        val_mse, val_mae = evaluate(model, val_data)
        history.append(val_mse)
        if val_mse < best_val - min_delta:        # "minimum improvement" rule
            best_val, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        if verbose:
            print("epoch %3d  val_mse=%8.3f  val_mae=%6.3f  best=%8.3f  wait=%d"
                  % (epoch, val_mse, val_mae, best_val, wait))
        if wait >= patience:
            print("Early stopping at epoch %d (best val_mse=%.3f)" % (epoch, best_val))
            break
    if best_state is not None:
        model.load_state_dict(best_state)         # restore_best_weights=True
    return history


# ---- Robust data loader: identical behaviour in Colab and local Jupyter ----
# A file already in this folder is used as-is; a missing file is downloaded and
# its contents are validated. Pure Python (no shell), so it behaves the same in
# Colab, local Jupyter, Windows/Mac/Linux.
import urllib.request

DATA_SOURCES = {
    "training_data.csv": [
        "https://rth.dk/internal/index.php/s/S4jQMaER6nYAJGe/download",
    ],
    "validation_data.csv": [
        "https://rth.dk/internal/index.php/s/oHspJCgniRMog6r/download",
    ],
}

def _is_valid_csv(path):
    """A real data file starts with the known header, not an HTML error page."""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            first = fh.readline()
        return ("target" in first) and ("deltaGb" in first)
    except OSError:
        return False

def fetch_data(fname, dest_dir="."):
    """Return the path to a valid `fname`, downloading it only if needed.
    urllib raises on HTTP errors (unlike a bare `curl -o`) and we re-check the
    content, so a bad/expired URL fails loudly instead of silently writing an
    HTML page into a .csv."""
    dest = os.path.join(dest_dir, fname)
    if _is_valid_csv(dest):
        print("using existing", dest)
        return dest
    problems = []
    for url in DATA_SOURCES[fname]:
        try:
            print("downloading %s from %s ..." % (fname, url.split("/")[2]))
            urllib.request.urlretrieve(url, dest)
        except Exception as e:
            problems.append("%s -> %s" % (url, e))
            continue
        if _is_valid_csv(dest):
            return dest
        problems.append("%s -> downloaded file is not a valid CSV (an error page?)" % url)
    if os.path.exists(dest):
        os.remove(dest)                       # never leave a corrupt .csv behind
    raise RuntimeError(
        "Could not obtain %s. Upload it into this folder manually, or fix the "
        "URLs in DATA_SOURCES above.\nTried:\n  %s" % (fname, "\n  ".join(problems)))

fetch_data("training_data.csv")
fetch_data("validation_data.csv")


def load_state(path):
    """torch.load that keeps weights_only=True on torch>=1.13 (safer) but
    still works on older torch versions that lack that argument."""
    try:
        return torch.load(path, weights_only=True)
    except TypeError:
        return torch.load(path)


# Training / validation data
PATH = './'
d = pd.read_csv(PATH + 'training_data.csv').values.tolist()
dv = pd.read_csv(PATH + 'validation_data.csv').values.tolist()
(x30, g, y) = preprocess_seq(d)
(x30v, gv, yv) = preprocess_seq(dv)
x30_t, g_t, y_t = to_tensors(x30, g, y)
x30v_t, gv_t, yv_t = to_tensors(x30v, gv, yv)

print('Data loaded')


Using device: cuda
downloading training_data.csv from rth.dk ...
downloading validation_data.csv from rth.dk ...
Data loaded


## Model definition and baseline performance

---


In the code below, we will re-use the simplified version of the CRISPRon ontarget model we created in the previous exercise, but we will preprocess the data in different ways, by masking out information from the input in order to glean its relative importance.

In [2]:
DROPOUT_DENSE = 0.3
CONV_1_SIZE = 3
N_CONV_1 = 40
N_DENSE = 40
N_OUT = 40

class SimpleCRISPRon(nn.Module):
    """A simplified version of the CRISPRon on-target model (PyTorch).

    One 1D convolution over the one-hot sequence, followed by fully connected
    (dense) layers. The binding energy dGb is concatenated in *after* the first
    dense layer.
    """
    def __init__(self, n_conv=N_CONV_1, kernel=CONV_1_SIZE, n_dense=N_DENSE,
                 n_out=N_OUT, dropout=DROPOUT_DENSE, seq_len=eLENGTH30, depth=eDEPTH):
        super().__init__()
        self.conv = nn.Conv1d(depth, n_conv, kernel)      # (B,4,30) -> (B,n_conv,28)
        conv_out_len = seq_len - kernel + 1               # 30 - 3 + 1 = 28
        flat = n_conv * conv_out_len                      # flattened conv features
        self.collect = nn.Linear(flat, n_dense)           # "dense_0"
        self.dense1 = nn.Linear(n_dense + 1, n_dense)     # "dense_1"  (+1 = raw dGb)
        self.dense2 = nn.Linear(n_dense, n_out)           # "dense_2"
        self.dense3 = nn.Linear(n_out, n_out)             # "dense_on_off"
        self.out = nn.Linear(n_out, 1)                    # output (linear, no activation)
        self.drop = nn.Dropout(dropout)
        self.apply(self._xavier_uniform_init)

    def features(self, x, g):
        # Conv1d wants (batch, channels, length); our one-hot is
        # (batch, length=30, channels=4), so swap the last two axes.
        x = x.permute(0, 2, 1)                            # (B,30,4) -> (B,4,30)
        z = torch.relu(self.conv(x))                      # convolution + ReLU
        z = torch.flatten(z, start_dim=1)                 # (B, n_conv*28)
        z = self.drop(torch.relu(self.collect(z)))        # dense_0 + ReLU + dropout
        z = torch.cat([g, z], dim=1)                      # concat raw dGb
        z = self.drop(torch.relu(self.dense1(z)))         # dense_1
        z = self.drop(torch.relu(self.dense2(z)))         # dense_2
        z = self.drop(torch.relu(self.dense3(z)))         # dense_on_off
        return z

    def forward(self, x, g):
        return self.out(self.features(x, g))              # (B, 1)

    @staticmethod
    def _xavier_uniform_init(m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

model = SimpleCRISPRon().to(device)
print(model)
print('Model defined')


SimpleCRISPRon(
  (conv): Conv1d(4, 40, kernel_size=(3,), stride=(1,))
  (collect): Linear(in_features=1120, out_features=40, bias=True)
  (dense1): Linear(in_features=41, out_features=40, bias=True)
  (dense2): Linear(in_features=40, out_features=40, bias=True)
  (dense3): Linear(in_features=40, out_features=40, bias=True)
  (out): Linear(in_features=40, out_features=1, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
)
Model defined


### Preprocessing

First we preprocess the data exactly as in exercise 1, then train once to produce the
starting weights that every masking experiment below will reuse.

Expect the same ballpark as exercise 1: validation **MSE roughly 160-190, MAE roughly
10-11**. You will not reproduce your exercise-1 number exactly - the random
initialisation, the dropout masks and the batch order differ from run to run, and that
alone moves the MSE by about 10.


In [3]:
# preprocess (no mask) and convert to tensors
(x30, g, y) = preprocess_seq(d)
(x30v, gv, yv) = preprocess_seq(dv)
x30_t, g_t, y_t = to_tensors(x30, g, y)
x30v_t, gv_t, yv_t = to_tensors(x30v, gv, yv)
#print(x30[0], g[0], y[0])

print("training...")
LEARN = 1e-3
EPOCHS = 200
BATCH_SIZE = 64

# save the initial (untrained) weights so every masking experiment below can
# restart from exactly the same starting point
# (re-)create a FRESH model here so re-running this cell always snapshots
# UNTRAINED weights. Otherwise a second run would save the already-trained
# weights as the "initial" state and silently corrupt the experiments below.
model = SimpleCRISPRon().to(device)
init_state = copy.deepcopy(model.state_dict())
torch.save(init_state, 'model_30.init.pt')

history = train(model, (x30_t, g_t, y_t), (x30v_t, gv_t, yv_t),
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
                patience=25, min_delta=0.1)

print("done")
print("evaluating validation data using best model weights")
print(evaluate(model, (x30v_t, gv_t, yv_t)))


training...
epoch   0  val_mse= 337.918  val_mae=14.750  best= 337.918  wait=0
epoch   1  val_mse= 249.929  val_mae=12.589  best= 249.929  wait=0
epoch   2  val_mse= 241.293  val_mae=12.496  best= 241.293  wait=0
epoch   3  val_mse= 229.638  val_mae=12.174  best= 229.638  wait=0
epoch   4  val_mse= 262.719  val_mae=13.188  best= 229.638  wait=1
epoch   5  val_mse= 277.083  val_mae=13.638  best= 229.638  wait=2
epoch   6  val_mse= 241.703  val_mae=12.564  best= 229.638  wait=3
epoch   7  val_mse= 288.185  val_mae=13.941  best= 229.638  wait=4
epoch   8  val_mse= 297.940  val_mae=14.242  best= 229.638  wait=5
epoch   9  val_mse= 280.119  val_mae=13.743  best= 229.638  wait=6
epoch  10  val_mse= 269.170  val_mae=13.456  best= 229.638  wait=7
epoch  11  val_mse= 257.327  val_mae=13.109  best= 229.638  wait=8
epoch  12  val_mse= 224.715  val_mae=12.122  best= 224.715  wait=0
epoch  13  val_mse= 303.872  val_mae=14.431  best= 224.715  wait=1
epoch  14  val_mse= 271.310  val_mae=13.511  best=

## Exercise 2.1

#### Masking out input information

A neural network does not report which inputs it relies on. The workaround: delete a piece of the input, retrain, and see how much the validation error grows.

The cell below has two knobs.

**`mask`** - a list of 30 values, one per position of the 30-mer. `0` keeps the base, `1` masks it out (that position's one-hot vector is left all zeros, so the model sees "no base here"). Positions:

```
0-3         4-23          24-26        27-29
4nt prefix  20nt spacer   3nt NGG PAM  3nt suffix
```

For example, masking the 4-nt prefix:

```
mask = [1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
```

**`use_dgb`** - `False` sets ΔGb to 0 for every guide. The architecture does not change; the feature just carries no information.

Both knobs are applied to the training **and** validation data, and the cell retrains from the saved initial weights - so each run answers "how well can the model do if this information never existed?"

Compare the printed validation MSE against the unmasked baseline above. Re-run the whole cell after every edit, and repeat a run before trusting a small difference: run-to-run noise alone is several MSE.

In [4]:
mask = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
use_dgb = True

# training / validation preprocess with the mask / dGb toggle
(x30, g, y) = preprocess_seq(d, mask, use_dgb)
(x30v, gv, yv) = preprocess_seq(dv, mask, use_dgb)
x30_t, g_t, y_t = to_tensors(x30, g, y)
x30v_t, gv_t, yv_t = to_tensors(x30v, gv, yv)
#print(x30[0], g[0], y[0])

# restore the initial weights (train() builds a fresh Adam each call, so every
# experiment truly starts from the same point)
if os.path.exists('model_30.init.pt'):
    print('weights loaded')
    model.load_state_dict(load_state('model_30.init.pt'))
else:
    raise RuntimeError("model_30.init.pt not found - run the baseline training cell above first.")

history = train(model, (x30_t, g_t, y_t), (x30v_t, gv_t, yv_t),
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
                patience=25, min_delta=0.1)

print("done")
print("evaluating validation data using best model weights")
print(evaluate(model, (x30v_t, gv_t, yv_t)))


weights loaded
epoch   0  val_mse= 311.495  val_mae=14.112  best= 311.495  wait=0
epoch   1  val_mse= 266.764  val_mae=13.169  best= 266.764  wait=0
epoch   2  val_mse= 253.800  val_mae=12.876  best= 253.800  wait=0
epoch   3  val_mse= 252.169  val_mae=12.880  best= 252.169  wait=0
epoch   4  val_mse= 256.916  val_mae=13.097  best= 252.169  wait=1
epoch   5  val_mse= 220.230  val_mae=12.013  best= 220.230  wait=0
epoch   6  val_mse= 322.735  val_mae=14.947  best= 220.230  wait=1
epoch   7  val_mse= 228.317  val_mae=12.277  best= 220.230  wait=2
epoch   8  val_mse= 228.440  val_mae=12.290  best= 220.230  wait=3
epoch   9  val_mse= 237.545  val_mae=12.483  best= 220.230  wait=4
epoch  10  val_mse= 215.566  val_mae=11.875  best= 215.566  wait=0
epoch  11  val_mse= 223.420  val_mae=12.134  best= 215.566  wait=1
epoch  12  val_mse= 261.775  val_mae=13.302  best= 215.566  wait=2
epoch  13  val_mse= 284.815  val_mae=13.974  best= 215.566  wait=3
epoch  14  val_mse= 211.494  val_mae=11.763  be

### Exercise 2.1.1

*(Optional - if time allows.)*

Measure how much ΔGb is worth. Train **both** settings, `use_dgb = True` and `use_dgb = False`, 3-5 times each - and give the two members of each pair the same seed, so they see identical starting weights, dropout masks and batch order:

```python
n = 3
for seed in range(n):
    for use_dgb in (True, False):
        m = SimpleCRISPRon().to(device)
        m.load_state_dict(load_state('model_30.init.pt'))   # identical starting weights
        torch.manual_seed(seed)                             # identical dropout / batch order
        # preprocess with use_dgb, train m, evaluate, record the validation MSE
```

(Training a fresh `m` inside the loop leaves the notebook's `model` untouched.)

1. For each seed, compute `MSE(without ΔGb) - MSE(with ΔGb)`. Report the differences and their median. Why is comparing within a seed better than comparing the two averages?
2. The full CRISPRon model gets mse 141.3 with ΔGb and 145.1 without. Is your gap smaller or larger than its 3.8?
3. Explain the size of your gap. Use this fact: ΔGb is the binding energy of the 20-nt spacer:target duplex, so it is computed from the sequence the model is already given.


In [5]:
# --- Exercise 2.1.1: how much is dGb worth? (paired design) ---
def run_dgb(use_dgb, seed):
    """One training run. Locals only, so the notebook's globals are left alone."""
    (x30, g, y)    = preprocess_seq(d,  None, use_dgb)
    (x30v, gv, yv) = preprocess_seq(dv, None, use_dgb)
    m = SimpleCRISPRon().to(device)
    m.load_state_dict(load_state('model_30.init.pt'))   # identical starting weights
    torch.manual_seed(seed)                             # identical dropout / batch order
    train(m, to_tensors(x30, g, y), to_tensors(x30v, gv, yv),
          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
          patience=25, min_delta=0.1, verbose=False)
    return evaluate(m, to_tensors(x30v, gv, yv))[0]

diffs = []
for seed in range(3):                                   # raise to 5 if time allows
    on  = run_dgb(True,  seed)
    off = run_dgb(False, seed)
    diffs.append(off - on)
    print("seed %d: with dGb %6.1f | without %6.1f | diff %+6.1f"
          % (seed, on, off, off - on))
print("median diff = %+.1f MSE" % float(np.median(diffs)))

# 1. Why compare within a seed instead of comparing the two averages?
#    Both members of a pair see the SAME initial weights, the SAME dropout masks and
#    the SAME batch order, so everything except dGb is held fixed and the shared
#    noise cancels in the difference. Between seeds that noise is about 10 MSE -
#    larger than the effect we are trying to measure - so two averages of
#    independent runs would drown the signal we are after.
#
# 2. The full CRISPRon gap is 145.1 - 141.3 = 3.8 MSE. The median gap here is the
#    same order of magnitude: a few MSE. dGb helps, but only a little, in both.
#
# 3. Why is the gap so small? dGb is the binding energy of the 20-nt spacer:target
#    duplex, i.e. a deterministic FUNCTION of sequence the convolution already sees.
#    It is not new information - it is a hand-crafted summary of information the
#    model already has. Setting it to 0 removes a shortcut, not a data source, and
#    the network re-derives most of it from the one-hot input. (A single scalar also
#    cannot carry much next to a 30x4 sequence input.)
#
# Note: an effect of a few MSE is inside run-to-run noise, so the honest conclusion
# from 3 seeds is "small, and not sharply resolved". Report the median, not the best
# run, and increase the number of seeds if you want a tighter answer.


Early stopping at epoch 72 (best val_mse=159.675)
Early stopping at epoch 51 (best val_mse=176.253)
seed 0: with dGb  159.7 | without  176.3 | diff  +16.6
Early stopping at epoch 57 (best val_mse=160.049)
Early stopping at epoch 46 (best val_mse=165.648)
seed 1: with dGb  160.0 | without  165.6 | diff   +5.6
Early stopping at epoch 74 (best val_mse=157.848)
Early stopping at epoch 45 (best val_mse=218.757)
seed 2: with dGb  157.8 | without  218.8 | diff  +60.9
median diff = +16.6 MSE


### Exercise 2.1.2

*(Optional - if time allows.)*

Modify the code above to mask out part of the ontarget sequence.

What is the effect of masking out the GG of the NGG PAM?

What is the effect of masking out all three nucleotides of the PAM?

What is the effect of masking out first base just before the PAM?

Does your answers depend on whether ΔGb is included in the model? For which of the three questions above would it be appropriate / inappropriate to include the energy parameter ΔGb?

If time allows, come up with other parts of the sequence to mask out and check your expectation of the impact with the actual result?


In [6]:
# --- Exercise 2.1.2: mask parts of the sequence ---
def run_mask(mask, use_dgb=True, seed=0):
    """One training run. Locals only, so the notebook's globals are left alone."""
    (x30, g, y)    = preprocess_seq(d,  mask, use_dgb)
    (x30v, gv, yv) = preprocess_seq(dv, mask, use_dgb)
    m = SimpleCRISPRon().to(device)
    m.load_state_dict(load_state('model_30.init.pt'))   # identical starting weights
    torch.manual_seed(seed)                             # identical dropout / batch order
    train(m, to_tensors(x30, g, y), to_tensors(x30v, gv, yv),
          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
          patience=25, min_delta=0.1, verbose=False)
    return evaluate(m, to_tensors(x30v, gv, yv))[0]

# 0-indexed layout: prefix 0-3, spacer 4-23, PAM 24-26 (N=24, GG=25 and 26),
# suffix 27-29. "The base just before the PAM" is position 23 - the LAST base of
# the 20-nt spacer.
no_mask  = [0]*30
mask_gg  = [0]*25 + [1, 1] + [0]*3        # GG of the NGG PAM  (not part of the spacer)
mask_pam = [0]*24 + [1, 1, 1] + [0]*3     # all 3 PAM bases    (not part of the spacer)
mask_pre = [0]*23 + [1] + [0]*6           # position 23        (IS part of the spacer)

# Block A - masking the PAM. dGb is computed from the spacer only, so it is
# independent of the PAM and may stay ON. Reference = baseline WITH dGb.
base_on = run_mask(no_mask, use_dgb=True)
print("baseline    (+dGb) MSE = %6.1f" % base_on)
for name, mk in (("mask GG", mask_gg), ("mask NGG", mask_pam)):
    v = run_mask(mk, use_dgb=True)
    print("%-11s (+dGb) MSE = %6.1f   delta = %+6.1f" % (name, v, v - base_on))

# Block B - masking a SPACER base. dGb is derived from the spacer, so leaving it ON
# would leak the masked base back in. Turn it OFF - and then the reference must ALSO
# be the no-dGb baseline, otherwise we would be measuring two changes at once.
base_off = run_mask(no_mask,  use_dgb=False)
v_pre    = run_mask(mask_pre, use_dgb=False)
print("baseline    (-dGb) MSE = %6.1f" % base_off)
print("mask pos 23 (-dGb) MSE = %6.1f   delta = %+6.1f" % (v_pre, v_pre - base_off))

# Interpretation:
#  - Masking the GG: no effect, and it CANNOT have one. Positions 25 and 26 are "GG"
#    in 100% of the guides, so they carry zero information for telling good guides
#    from bad ones. (The PAM is required for cutting, not for ranking efficiency.)
#  - Masking all three PAM bases additionally removes position 24, the "N", which
#    does vary (TGG / AGG / GGG / CGG). Any effect comes from that base alone, and
#    it is small.
#  - Masking position 23, the last spacer base: by far the largest of the three.
#    PAM-proximal spacer positions have real nucleotide preferences that drive
#    on-target efficiency.
#  - Is it appropriate to include dGb?
#      * mask GG, or the whole PAM -> YES, keep dGb: it is computed from the spacer,
#        so masking the PAM cannot leak through it.
#      * mask a spacer base        -> NO: dGb still "sees" that base and feeds it
#        back in, hiding the base's true importance. Use use_dgb=False, and compare
#        against the no-dGb baseline, as block B does.
#  - A delta of a few MSE is inside run-to-run noise (about 10 MSE between seeds).
#    Repeat with several seeds before believing a small one.


Early stopping at epoch 79 (best val_mse=156.353)
baseline    (+dGb) MSE =  156.4
Early stopping at epoch 67 (best val_mse=160.181)
mask GG     (+dGb) MSE =  160.2   delta =   +3.8
Early stopping at epoch 71 (best val_mse=160.109)
mask NGG    (+dGb) MSE =  160.1   delta =   +3.8
Early stopping at epoch 49 (best val_mse=168.845)
Early stopping at epoch 41 (best val_mse=229.921)
baseline    (-dGb) MSE =  168.8
mask pos 23 (-dGb) MSE =  229.9   delta =  +61.1


## Exercise 2.2

Suggest other features to include in the model, for example epigentic markers or adjusting the model for a particular type of CRISPR experiment?

How would you incorporate them into the model?

What could these features mean for the precision and generalizability of the model?


In [7]:
# --- Exercise 2.2 (open-ended) ---
# Other features and HOW to add them:
#  - Chromatin accessibility (ATAC-/DNase-seq), nucleosome occupancy: Cas9 must
#    physically reach the DNA, so open chromatin -> higher efficiency. Add as
#    extra scalar input(s) concatenated at the dGb concat point (torch.cat).
#  - DNA methylation / histone marks near the target site.
#  - Experiment metadata (Cas9 variant, delivery, cell type): one-hot or an
#    embedding, concatenated the same way.
#  - Extra sequence context, GC content, secondary-structure / melting temp.
#
# Effect on precision & generalizability:
#  - Relevant features can raise precision (lower error) for the setting they
#    describe.
#  - BUT epigenetic features are CELL-TYPE SPECIFIC, so a model trained with them
#    on one cell type may generalise WORSE to others; and more inputs raise
#    overfitting risk when data is limited. There is a trade-off between fitting a
#    specific experimental context and staying general.


## Reference

These exercises use a **simplified** version of the CRISPRon on-target efficiency model:

- Xiang, X., Corsi, G. I., Anthon, C., *et al.* (2021). Enhancing CRISPR-Cas9 gRNA efficiency prediction by data integration and deep learning. *Nature Communications*, **12**, 3238. https://doi.org/10.1038/s41467-021-23576-0
